In [ ]:
# Import Necessary Packages/Modules
import time, warnings
import numpy as np
import pandas as pd
from scipy import stats
from skeLCS import eLCS   # pip install scikit-eLCS
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectFromModel
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate, train_test_split
from sklearn.metrics import balanced_accuracy_score, f1_score, confusion_matrix

#Reusable constants.  You can change these to match your own data and preferences.
DATA_FILE  = "medical_insurance_cleaned_v2.csv"   # output of Task 3
SEED       = 42
N_SPLITS   = 3      # same 3-fold CV as the baseline
N_REPEATS  = 1      # raise to 5 for a meaningful significance test (15 scores per model, much slower)
N_FEATURES = 4      # features kept by the selector
N_JOBS     = -1     # folds run in parallel; set to 1 if you hit memory/pickling problems

In [ ]:
# Define columns that may introduce data leakage
#These values are derived from the target variable and should not be used as features in the model to avoid data leakage
LEAKAGE_COLS = ["is_high_risk", "annual_premium", "monthly_premium", "annual_medical_cost",
                "claims_count", "avg_claim_amount", "total_claims_paid"]

#converting the risk_score into 3 bands (low, medium, high) using quantiles and storing it in a new column called risk_band
df = pd.read_csv(DATA_FILE, keep_default_na=False, na_values=[""])   # keeps "None" in alcohol_freq as a category
y = pd.qcut(df["risk_score"], 3, labels=False).values                # 0 = low, 1 = medium, 2 = high
BAND_NAMES = {0: "Low", 1: "Medium", 2: "High"}

# Prepare the feature matrix X by dropping the leakage columns, risk_score, and person_id. Convert categorical columns to integer codes for model compatibility.
X_df = df.drop(columns=LEAKAGE_COLS + ["risk_score", "person_id"])
for c in X_df.select_dtypes(exclude="number"):
    X_df[c] = X_df[c].astype("category").cat.codes                   # text -> integer codes
feature_names = list(X_df.columns)
X = X_df.values.astype(float)
print("X:", X.shape, "| class shares:", np.round(np.bincount(y) / len(y), 3))

In [3]:
# Define a feature selector function that uses a shallow decision tree to select the N_FEATURES most important features. This function will be used in the pipeline to ensure that feature selection is performed on each training fold without leaking information from the test fold.
def selector():
    # keeps the N_FEATURES most important columns according to a shallow tree.
    # Inside a Pipeline it is refit on each training fold only, so the test fold never influences selection.
    return SelectFromModel(DecisionTreeClassifier(max_depth=5, random_state=SEED),
                           max_features=N_FEATURES, threshold=-np.inf)

configs = {
    "A. Original eLCS, preprocessed data (45 features)": Pipeline([
        ("elcs", eLCS(random_state=SEED))]),
    "B. + feature selection": Pipeline([
        ("select", selector()),
        ("elcs", eLCS(random_state=SEED))]),
    "C. + MinMax scaling": Pipeline([
        ("select", selector()), ("scale", MinMaxScaler()),
        ("elcs", eLCS(random_state=SEED))]),
    "D. + correct-set subsumption": Pipeline([
        ("select", selector()), ("scale", MinMaxScaler()),
        ("elcs", eLCS(random_state=SEED, do_correct_set_subsumption=True))]),
    "E. Improved (+ 20k iterations, N=1500)": Pipeline([
        ("select", selector()), ("scale", MinMaxScaler()),
        ("elcs", eLCS(random_state=SEED, do_correct_set_subsumption=True,
                      learning_iterations=20000, N=1500))]),
}
IMPROVED = "E. Improved (+ 20k iterations, N=1500)"
BASELINE = "A. Original eLCS, preprocessed data (45 features)"

In [ ]:
# Run cross-validation for each configuration and collect scores and execution times. The scoring metrics used are balanced accuracy and macro F1 score. The results are printed for each configuration, showing the mean and standard deviation of the balanced accuracy, the mean macro F1 score, and the time taken for cross-validation.
cv = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=SEED)
scoring = {"bal_acc": "balanced_accuracy", "f1_macro": "f1_macro"}  #scoring metrics to be used for cross-validation

scores, times = {}, {} #containers to hold the cross-validation scores and execution times for each configuration
for name, pipe in configs.items():
    t0 = time.time()
    res = cross_validate(pipe, X, y, cv=cv, scoring=scoring, n_jobs=N_JOBS)
    scores[name] = res
    times[name] = time.time() - t0
    print(f"{name}\n   balanced acc {res['test_bal_acc'].mean():.3f} +/- {res['test_bal_acc'].std():.3f}"
          f" | macro-F1 {res['test_f1_macro'].mean():.3f} | {times[name]:.0f}s")

In [4]:
# Define what the leaky columns are
LEAKAGE_COLS = ["is_high_risk", "annual_premium", "monthly_premium", "annual_medical_cost", "claims_count", "avg_claim_amount", "total_claims_paid",]

# Excludes the ones columns that come as a result of "risk_score" along with the IDs
X = df.drop(columns=LEAKAGE_COLS + ["risk_score", "risk_band", "person_id"])

In [5]:
# eLCS needs numbers, so every text column becomes integer codes
for col in X.select_dtypes(exclude="number"):
    X[col] = X[col].astype("category").cat.codes

In [6]:
# Converts the table to a plain array of decimals
X = X.values.astype(float)

In [7]:
%%time
# Run and score model
model = eLCS(random_state=42) # Seeded model with a state to make results repeatable
cv = StratifiedKFold(3, shuffle=True, random_state=42) # Sets up 3-fold cross validation
baseline_scores = cross_val_score(model, X, y, cv=cv, scoring="balanced_accuracy") # Runs the test loop 3 times and returns 3 scores with the scoring set to balanced accuracy
print(f"Baseline eLCS (45 features), CV balanced accuracy: {baseline_scores.mean():.3f}")
print(f"Fold scores: {np.round(baseline_scores, 3)}")

Baseline eLCS (45 features), CV balanced accuracy: 0.568
Fold scores: [0.505 0.641 0.56 ]
CPU times: total: 2min 59s
Wall time: 3min 30s


In [8]:
%%time
# Fresh model trained on entire dataset
model = eLCS(random_state=42)
model.fit(X, y)
print(f"Training accuracy: {model.get_final_accuracy():.3f}") # How accurate is is the final set of rules
print(f"Instance coverage: {model.get_final_instance_coverage():.1%}") # Percentage of training rows covered by at least one rule

Training accuracy: 0.488
Instance coverage: 74.5%
CPU times: total: 3min 41s
Wall time: 4min 41s


In [9]:
# Comparison with decision tree
tree = DecisionTreeClassifier(max_depth=5, random_state=42) # Shallow decision tree, kept seeded for consistency
tree_scores = cross_val_score(tree, X, y, cv=cv, scoring="balanced_accuracy") # 3-fold cross validation with the scoring set to balanced accuracy
print(f"Decision tree reference, CV balanced accuracy: {tree_scores.mean():.3f}")

Decision tree reference, CV balanced accuracy: 0.953


In [10]:
#Rerunning previous parts of code to get the labels back for a check
df = pd.read_csv("medical_insurance.csv", keep_default_na=False, na_values=[""])

df["risk_band"] = pd.qcut(df["risk_score"], 3, labels=False)
y = df["risk_band"].values

LEAKAGE_COLS = ["is_high_risk", "annual_premium", "monthly_premium", "annual_medical_cost", "claims_count", "avg_claim_amount", "total_claims_paid",]

X = df.drop(columns=LEAKAGE_COLS + ["risk_score", "risk_band", "person_id"])

for col in X.select_dtypes(exclude="number"):
    X[col] = X[col].astype("category").cat.codes

# Checking for top features
feature_names = X.columns   # Get the column names of the features
tree.fit(X, y)  # Train a decision tree on the full dataset
print(pd.Series(tree.feature_importances_, index=feature_names).sort_values(ascending=False).head(10))  # feature_importances_ gives one number per feature that add up to 1

# Converts the table back into to a plain array of decimals for future use
X = X.values.astype(float)

chronic_count     0.404078
age               0.366792
smoker            0.160757
bmi               0.068372
sex               0.000000
income            0.000000
urban_rural       0.000000
education         0.000000
marital_status    0.000000
household_size    0.000000
dtype: float64


In [11]:
%%time
# eLCS with only the top 4 features
SELECTED = ["chronic_count", "age", "smoker", "bmi"]

# Builds a new table with just those columns, then encodes them
X = df[SELECTED].copy()
for col in X.select_dtypes(exclude="number"):
    X[col] = X[col].astype("category").cat.codes
X = X.values.astype(float)

model = eLCS(random_state=42)
cv = StratifiedKFold(3, shuffle=True, random_state=42)
selected_scores = cross_val_score(model, X, y, cv=cv)
print(f"eLCS (4 features), CV balanced accuracy: {selected_scores.mean():.3f}")
print(f"Fold scores: {np.round(selected_scores, 3)}")

eLCS (4 features), CV balanced accuracy: 0.827
Fold scores: [0.821 0.862 0.799]
CPU times: total: 1min 38s
Wall time: 2min 18s


In [12]:
# Printing the parameters used for our tests
print(eLCS(random_state=42).get_params())

{'N': 1000, 'acc_sub': 0.99, 'beta': 0.2, 'chi': 0.8, 'delta': 0.1, 'discrete_attribute_limit': 10, 'do_GA_subsumption': True, 'do_correct_set_subsumption': False, 'fitness_reduction': 0.1, 'init_fit': 0.01, 'learning_iterations': 10000, 'match_for_missingness': False, 'mu': 0.04, 'nu': 5, 'p_spec': 0.5, 'random_state': 42, 'reboot_filename': None, 'selection_method': 'tournament', 'specified_attributes': array([], dtype=float64), 'theta_GA': 25, 'theta_del': 20, 'theta_sel': 0.5, 'theta_sub': 20, 'track_accuracy_while_fit': False}


In [13]:
results = pd.DataFrame({
    "Model": ["eLCS baseline (45 features)", "Decision tree (reference)", "eLCS (4 features)"],
    "CV balanced accuracy": [baseline_scores.mean(), tree_scores.mean(), selected_scores.mean()],
}).round(3)
results

,Model,CV balanced accuracy
0,eLCS baseline (45 features),0.568
1,Decision tree (reference),0.953
2,eLCS (4 features),0.827
